# Fine-tune DeReC on DRAGON — RuModernBERT-small

Train a RuModernBERT-small classifier to distinguish **grounded** from **ungrounded** RAG answers.

- Dataset: DRAGON (12600 train + 2700 val + 2700 test)
- Base model: `deepvk/RuModernBERT-small` (max_pos=8192)
- Binary classification: grounded=1, ungrounded=0
- Input format: `{question} [SEP] {answer} [SEP] {e1} [SEP] {e2}` (structured, native [SEP])
- max_length=2048 — covers ~96% examples without truncation (vs 48% at 512)

## Setup

In [ ]:
# ! rm -r rag_fact_checking
# !git clone -b feature/evaluate-fact-checking https://github.com/BigMak1/rag_fact_checking.git
!git -C rag_fact_checking pull

In [ ]:
%%time
# %%capture

!uv pip install --system \
    accelerate==1.1.1 \
    datasets==3.1.0 \
    scikit-learn==1.5.2 \
    transformers==4.48.0 \
    sentence-transformers \
    tqdm \
    tiktoken \
    protobuf \
    einops \
    sentencepiece \
    bitsandbytes \
    huggingface_hub \
    faiss-gpu-cu12

In [ ]:
import sys
import os
import json

import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader
from transformers import AutoConfig, AutoModelForSequenceClassification
import matplotlib.pyplot as plt

sys.path.insert(0, "rag_fact_checking/DEREC")
from dataset_load import DatasetReader, UnifiedDataset, DATASET_CONFIGS
from main import EvidenceClassificationPipeline, generate_run_id

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

In [ ]:
CLASSIFIER_MODEL_NAME = "ruModernBert"
DATASET_NAME = "DRAGON"
BATCH_SIZE = 8
LEARNING_RATE = 5e-6
EPOCH_SIZE = 10
SEED = 44
MAX_LENGTH = 2048

HF_DATASET_REPO = "Makson4ic/dragon-derec-dataset"
HF_MODEL_REPO = "Makson4ic/derec-dragon-ruModernBert-small"
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

BASE_DIR = "rag_fact_checking/DEREC"
DATASET_PATH = os.path.join(BASE_DIR, "dataset", DATASET_NAME)

run_id = generate_run_id()
save_dir = os.path.join(
    BASE_DIR,
    f"saved_models/{DATASET_NAME.lower()}_classifier/run_{run_id}"
)
os.makedirs(save_dir, exist_ok=True)
print(f"Run ID: {run_id}")
print(f"Save dir: {save_dir}")

## Load Dataset

In [ ]:
# Load from HuggingFace Hub and save as local JSON for DatasetReader
ds = load_dataset(HF_DATASET_REPO)
os.makedirs(DATASET_PATH, exist_ok=True)

for split_name in ["train", "val", "test"]:
    records = list(ds[split_name])
    with open(os.path.join(DATASET_PATH, f"{split_name}.json"), "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False)
    print(f"{split_name}: {len(records)} examples")

# Load via DatasetReader
train_data = DatasetReader.read_dataset(DATASET_NAME, DATASET_PATH, "train")
val_data = DatasetReader.read_dataset(DATASET_NAME, DATASET_PATH, "val")
test_data = DatasetReader.read_dataset(DATASET_NAME, DATASET_PATH, "test")

# Check class balance
for name, split in [("train", train_data), ("val", val_data), ("test", test_data)]:
    labels = [item["label"] for item in split]
    n_grounded = sum(labels)
    n_ungrounded = len(labels) - n_grounded
    print(f"{name}: grounded={n_grounded}, ungrounded={n_ungrounded}")

In [ ]:
example = train_data[0]
print(f"Question: {example['question']}\n")
print(f"Answer: {example['answer']}\n")
print(f"Evidence texts ({len(example['evidence_texts'])} chunks):")
for i, e in enumerate(example['evidence_texts']):
    print(f"  [{i}] {e[:100]}...")
print(f"\nLabel: {example['label']}")

## Initialize Pipeline

In [ ]:
pipeline = EvidenceClassificationPipeline(
    dataset_name=DATASET_NAME,
    classifier_model=CLASSIFIER_MODEL_NAME,
)
print(f"input_format: {pipeline.input_format}")
print(f"max_length: {pipeline.max_length}")
print(f"sep_token: {pipeline.tokenizer.sep_token} (id={pipeline.tokenizer.sep_token_id})")

## Train

In [ ]:
%%time

pipeline.train(
    train_dataset=train_data,
    eval_dataset=val_data,
    batch_size=BATCH_SIZE,
    num_epochs=EPOCH_SIZE,
    learning_rate=LEARNING_RATE,
    save_dir=save_dir,
)

## Evaluate on Test Set

In [ ]:
# Load best checkpoint
best_model_path = os.path.join(save_dir, f"epoch_{pipeline.best_epoch}")
config = AutoConfig.from_pretrained(best_model_path, local_files_only=True)
pipeline.classifier = AutoModelForSequenceClassification.from_pretrained(
    best_model_path,
    config=config,
    local_files_only=True,
    trust_remote_code=True,
).to(pipeline.device)
print(f"Loaded best model from epoch {pipeline.best_epoch}")

# Prepare test data with structured format
test_claims, test_evidences, test_labels, test_q, test_a, test_ev = (
    pipeline.process_dragon_dataset(test_data)
)
test_dataset = UnifiedDataset(
    test_claims, test_evidences, test_labels, DATASET_NAME, pipeline.tokenizer,
    max_length=pipeline.max_length,
    input_format=pipeline.input_format,
    questions=test_q,
    answers=test_a,
    evidence_texts_list=test_ev,
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Single inference pass: collect logits and labels
pipeline.classifier.eval()
all_logits = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        inputs = {k: v.to(pipeline.device) for k, v in batch.items()}
        outputs = pipeline.classifier(**inputs)
        all_logits.append(outputs.logits.cpu())
        all_labels.append(inputs["labels"].cpu())

all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels).numpy()
all_preds = torch.argmax(all_logits, dim=1).numpy()

# Basic metrics
test_metrics = pipeline._calculate_metrics(all_labels, all_preds)
print("\nTest set metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(classification_report(all_labels, all_preds, target_names=["ungrounded", "grounded"]))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=["ungrounded", "grounded"]).plot(ax=ax, cmap="Blues")
ax.set_title("Confusion Matrix \u2014 RuModernBERT-small on DRAGON")
plt.tight_layout()
plt.show()

In [ ]:
error_probs = torch.softmax(all_logits, dim=-1)[:, 1].numpy()

fp_idx = np.where((all_labels == 0) & (all_preds == 1))[0]
fn_idx = np.where((all_labels == 1) & (all_preds == 0))[0]

print(f"False Positives (ungrounded \u2192 predicted grounded): {len(fp_idx)}")
print(f"False Negatives (grounded \u2192 predicted ungrounded): {len(fn_idx)}")

for name, indices in [("FALSE POSITIVE", fp_idx), ("FALSE NEGATIVE", fn_idx)]:
    print(f"\n{'='*80}")
    print(f"  {name} examples (showing up to 5)")
    print(f"{'='*80}")
    for i, idx in enumerate(indices[:5]):
        item = test_data[idx]
        evidence_short = item["evidence"][:300] + "..." if len(item["evidence"]) > 300 else item["evidence"]
        print(f"\n--- #{i+1} (idx={idx}, P(grounded)={error_probs[idx]:.4f}) ---")
        print(f"Claim: {item['claim']}")
        print(f"Evidence: {evidence_short}")
        print(f"True label: {all_labels[idx]}  Predicted: {all_preds[idx]}")

In [ ]:
# ROC-AUC from softmax probabilities
probs = torch.softmax(all_logits, dim=-1)[:, 1].numpy()

auc = roc_auc_score(all_labels, probs)
fpr, tpr, thresholds = roc_curve(all_labels, probs)

# Optimal threshold (Youden's J)
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
best_threshold = thresholds[best_idx]

print(f"ROC-AUC: {auc:.4f}")
print(f"Optimal threshold (Youden's J): {best_threshold:.4f}")
print(f"  TPR: {tpr[best_idx]:.4f}, FPR: {fpr[best_idx]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"ROC (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1, label="Random")
ax.scatter([fpr[best_idx]], [tpr[best_idx]], color="red", s=100, zorder=5,
           label=f"Optimal (t={best_threshold:.3f})")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve \u2014 RuModernBERT-small on DRAGON")
ax.legend(loc="lower right")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

In [ ]:
grounded_probs = probs[all_labels == 1]
ungrounded_probs = probs[all_labels == 0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(grounded_probs, bins=30, alpha=0.6, label="Grounded", color="steelblue")
ax.hist(ungrounded_probs, bins=30, alpha=0.6, label="Ungrounded", color="tomato")
ax.axvline(best_threshold, color="black", linestyle="--",
           label=f"Threshold={best_threshold:.3f}")
ax.set_xlabel("P(grounded)")
ax.set_ylabel("Count")
ax.set_title("Score Distribution: Grounded vs Ungrounded")
ax.legend()
plt.tight_layout()
plt.show()

## Save Results

In [ ]:
final_results = {
    "run_id": run_id,
    "best_epoch": pipeline.best_epoch,
    "test_metrics": test_metrics,
    "roc_auc": auc,
    "optimal_threshold": float(best_threshold),
    "config": {
        "classifier_model": CLASSIFIER_MODEL_NAME,
        "base_model": "deepvk/RuModernBERT-small",
        "dataset": DATASET_NAME,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCH_SIZE,
        "max_length": MAX_LENGTH,
        "input_format": "sep_structured",
        "seed": SEED,
    },
}

results_path = os.path.join(save_dir, "final_results.json")
with open(results_path, "w") as f:
    json.dump(final_results, f, indent=2)

print(f"Results saved to {results_path}")
print(json.dumps(final_results, indent=2))

## Publish to HuggingFace

In [ ]:
from huggingface_hub import HfApi

assert HF_TOKEN, "Set HF_TOKEN in the constants cell"

api = HfApi(token=HF_TOKEN)
checkpoint_dir = os.path.join(save_dir, f"epoch_{pipeline.best_epoch}")

api.create_repo(repo_id=HF_MODEL_REPO, exist_ok=True)
api.upload_folder(
    folder_path=checkpoint_dir,
    repo_id=HF_MODEL_REPO,
    repo_type="model",
)
print(f"Uploaded to https://huggingface.co/{HF_MODEL_REPO}")